In [5]:
import os
import json
import glob
import re
import numpy as np
import xarray as xr
from typing import Dict
from joblib import Parallel, delayed
from cftime import num2date, DatetimeNoLeap
from datetime import timedelta
import matplotlib.pyplot as plt

# optional deps
try:
    import xskillscore as xs
except Exception:
    xs = None
try:
    from scipy.stats import t as t_dist
except Exception:
    t_dist = None
    

In [6]:
class ErrorMetricCalculator:
    def __init__(self, regnam, tstart, tend, frequency,
                 model_list, ref_dict, exp_dict, path_in, out_path,
                 var_list=None, force=False,
                 ref_template="monthly/ERA5_analysis_monthly_{year}.nc",
                 mod_template="{year}_*.nc"):
        """
        regnam      : region key (see define_region()).
        tstart,tend : e.g. '2012-01-01', '2016-12-31' (year/month parsed).
        frequency   : {'monthly','3hourly','6hourly'} (monthly expected here).
        model_list  : list of experiment keys.
        ref_dict    : {'ERA5': {'run': '/path/to/ref'}}
        exp_dict    : {'EXP1': {'run': 'CASE_A', 'ref': 'ERA5'}, ...}
        path_in     : model path template with '%(CASENAME)', e.g. '/data/%(CASENAME)/monthly'
        out_path    : root directory for outputs.
        var_list    : e.g. ['U200','T','Z']; None -> defaults from extract_var_list().
        force       : overwrite outputs if True.
        ref_template: file pattern; accepts {year} and/or {var}.
        mod_template: file pattern; accepts {year} and/or {var}.
        """
        self.regnam = regnam
        self.tstart = tstart
        self.tend = tend
        self.frequency = frequency
        self.model_list = model_list
        self.ref_dict = ref_dict
        self.exp_dict = exp_dict
        self.path_in = path_in
        self.out_path = os.path.join(out_path, frequency)
        self.force = force
        self.ref_template = ref_template
        self.mod_template = mod_template

        self.var_dict = self.extract_var_list()
        self.var_list = var_list if var_list is not None else list(self.var_dict.keys())

        self.years = list(range(int(self.tstart[:4]), int(self.tend[:4]) + 1))

        self.seasons = {
            "DJF": ([12, 1, 2], "01-03"),
            "MAM": ([3, 4, 5], "04-06"),
            "JJA": ([6, 7, 8], "07-09"),
            "SON": ([9, 10, 11], "10-12"),
            "ANN": (list(range(1, 13)), "01-12"),
        }

        os.makedirs(self.out_path, exist_ok=True)

    # --------------------------- public API ---------------------------
    def compute(self, metric="zonal_bias", pool_across_years=True):
        if pool_across_years:
            for var in self.var_list:
                vinfo = self._require_varinfo(var)
                self._compute_zonal_mean_metric_period(metric, var, vinfo)
        else:
            for year in self.years:
                for var in self.var_list:
                    vinfo = self._require_varinfo(var)
                    self._compute_zonal_mean_metric_year(metric, var, vinfo, year, skip_first_year_djf=True)

    # -------------------------- period-pooled --------------------------    
    @staticmethod
    def _lev_name(da_or_ds):
        for k in ("plev", "lev", "level"):
            if k in getattr(da_or_ds, "dims", ()) or k in getattr(da_or_ds, "coords", {}):
                return k
        return None
    
    @staticmethod
    def _levels_to_hpa(coord):
        """Return (values_in_hPa, units_str). Tries to infer units robustly."""
        vals = np.array(coord.values, dtype=float)
        units = (getattr(coord, "attrs", {}) or {}).get("units", "") or str(getattr(coord, "units", ""))
        u = units.lower()
        # If units explicitly say Pa, convert to hPa
        if "pa" in u and "hpa" not in u:
            return vals / 100.0, "hPa"
        # Heuristic: very large numbers likely Pa
        if np.nanmax(np.abs(vals)) > 2000:
            return vals / 100.0, "hPa"
        # Otherwise assume already hPa
        return vals, "hPa"
    
    def _harmonize_vertical(self, obs, fcst, var, *, prefer="obs", tol_hpa=0.05):
        """
        Make obs and fcst share the same vertical coordinate (in hPa).
        - If var like 'U200' → your _select_level already reduced to 1 level; return as-is.
        - Else (no target level):
            • If both have a level dim → convert both to hPa; interp one onto the other's grid.
            • If only one has a level dim → raise (ask caller to use a target-plev var).
        Returns (obs_aligned, fcst_aligned, lev_name_or_None)
        """
        base, target_plev = self._parse_varname(var)
        if target_plev is not None:
            # Both streams should already be at a single (interpolated) level via _select_level()
            return obs, fcst, None
    
        lo = self._lev_name(obs)
        lf = self._lev_name(fcst)
    
        # No level dim on either side → nothing to match
        if lo is None and lf is None:
            return obs, fcst, None
    
        # If only one side has levels, safer to stop here
        if (lo is None) ^ (lf is None):
            side = "obs" if lo is not None else "fcst"
            raise ValueError(
                f"Vertical mismatch: {side} has a level dimension but the other does not. "
                f"Use a level-specific variable name (e.g., '{base}200') or ensure both "
                f"inputs carry compatible vertical coords."
            )
    
        # Both have level dims → convert to hPa and align/interp
        # Rename level coord temporarily if different names
        if lo != lf:
            # bring both to the same dim name (use obs' name)
            fcst = fcst.rename({lf: lo})
            lf = lo
    
        # Convert coord values to hPa (keep the name the same)
        obs_hpa, _ = self._levels_to_hpa(obs[lo])
        fcst_hpa, _ = self._levels_to_hpa(fcst[lo])
    
        # Attach hPa coords explicitly so we can interp in hPa space
        obs = obs.assign_coords({lo: obs_hpa})
        fcst = fcst.assign_coords({lo: fcst_hpa})
    
        # Decide reference grid
        ref = "obs" if prefer == "obs" else "fcst"
        if ref == "obs":
            # Interp fcst onto obs levels (within overlap only)
            # Build safe domain (min..max overlap)
            lo_min = max(np.nanmin(obs_hpa),  np.nanmin(fcst_hpa))
            lo_max = min(np.nanmax(obs_hpa),  np.nanmax(fcst_hpa))
            target = obs_hpa[(obs_hpa >= lo_min - tol_hpa) & (obs_hpa <= lo_max + tol_hpa)]
            if target.size == 0:
                raise ValueError("No overlapping pressure levels between obs and fcst.")
            fcst = fcst.interp({lo: target})
            obs  = obs.sel({lo: target})
        else:
            lo_min = max(np.nanmin(obs_hpa),  np.nanmin(fcst_hpa))
            lo_max = min(np.nanmax(obs_hpa),  np.nanmax(fcst_hpa))
            target = fcst_hpa[(fcst_hpa >= lo_min - tol_hpa) & (fcst_hpa <= lo_max + tol_hpa)]
            if target.size == 0:
                raise ValueError("No overlapping pressure levels between obs and fcst.")
            obs  = obs.interp({lo: target})
            fcst = fcst.sel({lo: target})
    
        # Stamp units for clarity
        obs[lo].attrs["units"]  = "hPa"
        fcst[lo].attrs["units"] = "hPa"
        return obs, fcst, lo

    def _compute_zonal_mean_metric_period(self, metric, var, vinfo):
        varin, vfac = vinfo['alias'], vinfo['fscl']
        season_list = ["DJF", "MAM", "JJA", "SON", "ANN"]
        period_lbl = self._period_label()

        for exp in self.model_list:
            ref = self.exp_dict[exp]['ref']
            out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_{metric}_{exp}_{period_lbl}.nc")
            if os.path.exists(out_file) and not self.force:
                print(f"Skipping {metric} for {var} in {exp} ({period_lbl}) — file exists.")
                continue
            if os.path.exists(out_file) and self.force:
                os.remove(out_file)

            bias_list, pval_list, sig_list = [], [], []
            mean_bias_list, rmse_list, pcorr_list, nmonths_list = [], [], [], []

            for season in season_list:
                time_sub = self._get_time_range_for_season_period(season)

                try:
                    obs = self._read_reference_data(
                        ref=ref, period=period_lbl, time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.ref_dict[ref]['run'], template=self.ref_template
                    )[varin].astype("float64")

                    ds = self._read_model_data(
                        exp=exp, period=period_lbl, time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run']),
                        template=self.mod_template
                    )
                    # strict time alignment (no silent nearest on wrong month)
                    obs = self._align_to_times(obs.to_dataset(name=varin), time_sub)[varin]
                    ds  = self._align_to_times(ds, time_sub)
                    fcst = ds[varin].astype("float64")
                except FileNotFoundError:
                    print(f"Data missing for {var}, {exp}, {period_lbl}, {season} — skipping.")
                    lat = xr.DataArray(np.full((1,), np.nan), dims=('lat',), coords={'lat':[np.nan]})
                    bias_list.append(xr.full_like(lat, np.nan).rename('bias').expand_dims(season=[season]))
                    pval_list.append(xr.full_like(lat, np.nan).rename('pvalue').expand_dims(season=[season]))
                    sig_list.append(xr.full_like(lat, 0).rename('sig_mask').expand_dims(season=[season]))
                    mean_bias_list += [np.nan]; rmse_list += [np.nan]; pcorr_list += [np.nan]; nmonths_list += [0]
                    continue

                lat_name = next((d for d in ['lat','latitude','y'] if d in obs.dims), None)
                lon_name = next((d for d in ['lon','longitude','x'] if d in obs.dims), None)
                if lat_name is None or lon_name is None:
                    raise ValueError("Expected lat/lon dims not found in data.")

                obs, fcst, lev_dim = self._harmonize_vertical(obs, fcst, var,prefer="fcst")
                
                # proceed; chunking is fine now, the two share the same vertical grid
                obs  = obs.chunk({d: -1 for d in obs.dims})
                fcst = fcst.chunk({d: -1 for d in fcst.dims})
                
                # time-by-lat zonal means
                obs_zm_t  = obs.mean(lon_name)
                fcst_zm_t = fcst.mean(lon_name)

                # seasonal mean fields
                obs_zm  = obs_zm_t.mean('time')
                fcst_zm = fcst_zm_t.mean('time')
                bias_zm = (fcst_zm - obs_zm).rename('bias')

                # significance on monthly samples with AR(1) Neff
                d_t   = (fcst_zm_t - obs_zm_t)
                N     = d_t.sizes.get('time', 1)
                nmonths_list.append(int(N))
                mean_d = d_t.mean('time')
                std_d  = d_t.std('time', ddof=1)

                r1 = xr.apply_ufunc(
                    self._lag1_corr_np, d_t,
                    input_core_dims=[['time']], output_core_dims=[[]],
                    vectorize=True, dask='parallelized', output_dtypes=[float]
                ).clip(min=-0.99, max=0.99)
                Neff = (N * (1.0 - r1) / (1.0 + r1)).clip(min=2.0, max=float(N))
                # FIX: enable dask in sqrt to avoid ValueError with chunked arrays
                sqrt_Neff = xr.apply_ufunc(np.sqrt, Neff, dask='parallelized', output_dtypes=[float])
                se   = xr.where(sqrt_Neff > 0, std_d / sqrt_Neff, np.nan)
                tstat = xr.where(se > 0, mean_d / se, np.nan)

                # elementwise df = max(1, Neff-1), dask-safe
                df = xr.apply_ufunc(
                    np.maximum, Neff - 1.0, 1.0,
                    dask='parallelized', output_dtypes=[float]
                )
                
                if t_dist is not None:
                    # two-sided p-value from Student-t, vectorized
                    pval = xr.apply_ufunc(
                        lambda t, df_: 2.0 * (1.0 - t_dist.cdf(np.abs(t), df=df_)),
                        tstat, df,
                        dask='parallelized', output_dtypes=[float]
                    )
                else:
                    # normal approximation fallback, vectorized
                    pval = xr.apply_ufunc(
                        lambda z: np.erfc(np.abs(z) / np.sqrt(2.0)),
                        tstat, dask='parallelized', output_dtypes=[float]
                    )
                pval = pval.rename('pvalue')
                sig_mask = (pval < 0.05).astype('i1').rename('sig_mask')

                # summary metrics (compute together to minimize graphs)
                w = self._coslat_weights_from_coords(obs[lat_name])
                
                # scalar 0-D DataArrays after reduction over latitude
                mb_da   = bias_zm.weighted(w).mean(lat_name)
                rmse_da = ((fcst_zm - obs_zm) ** 2).weighted(w).mean(lat_name) ** 0.5
                
                # convert to Python floats safely (works for dask-backed arrays)
                mean_bias = self._to_scalar_float(mb_da)
                rmse      = self._to_scalar_float(rmse_da)
                
                # pattern correlation across latitude
                if xs is not None:
                    try:
                        pcorr_da = xs.pearson_r(
                            obs_zm, fcst_zm,
                            dim=[lat_name],
                            weights=w,
                            skipna=True
                        )
                        pcorr = self._to_scalar_float(pcorr_da)
                    except TypeError:
                        pcorr_da = xs.pearson_r(obs_zm, fcst_zm, dim=[lat_name], skipna=True)
                        pcorr = self._to_scalar_float(pcorr_da)
                else:
                    pcorr = float(self._weighted_pearson_1d(obs_zm, fcst_zm, w))

                bias_list.append(bias_zm.expand_dims(season=[season]))
                pval_list.append(pval.expand_dims(season=[season]))
                sig_list.append(sig_mask.expand_dims(season=[season]))
                mean_bias_list.append(mean_bias); rmse_list.append(rmse); pcorr_list.append(pcorr)

            ds_out = xr.Dataset(
                data_vars=dict(
                    bias_zonal   = xr.concat(bias_list, dim='season'),
                    pvalue       = xr.concat(pval_list, dim='season'),
                    sig_mask     = xr.concat(sig_list, dim='season'),
                    mean_bias    = xr.DataArray(np.asarray(mean_bias_list), dims="season", coords={"season": season_list}),
                    rmse         = xr.DataArray(np.asarray(rmse_list),      dims="season", coords={"season": season_list}),
                    pattern_corr = xr.DataArray(np.asarray(pcorr_list),     dims="season", coords={"season": season_list}),
                    n_months     = xr.DataArray(np.asarray(nmonths_list),   dims="season", coords={"season": season_list}),
                )
            )
            self._annotate_metadata(ds_out, var)
            ds_out.to_netcdf(out_file)
            print(f"Saved: {out_file}")

    # ---------------------------- per-year -----------------------------
    def _to_scalar_float(self, da):
        """Reduce remaining dims by mean -> compute/load -> return a Python float."""
        if hasattr(da, "dims"):
            if da.ndim > 0:
                da = da.mean(dim=list(da.dims), skipna=True)
            da = da.load()                 # works for NumPy or Dask-backed
            return float(da.values.item()) # 0-D ndarray -> Python float
        # Fallback for plain arrays
        arr = np.asarray(da)
        if arr.size > 1:
            arr = np.nanmean(arr)
        return float(np.asarray(arr).item())
        
    def _collapse_except(self, da, keep):
        """Mean-reduce all dims not in `keep` (skipna=True)."""
        for d in list(da.dims):
            if d not in keep:
                da = da.mean(d, skipna=True)
        return da
    
    def _to_scalar_da(self, da):
        """Mean-reduce all remaining dims (skipna=True) so result is 0-D DataArray."""
        for d in list(da.dims):
            da = da.mean(d, skipna=True)
        return da
    
    def _compute_zonal_mean_metric_year(self, metric, var, vinfo, year, skip_first_year_djf=True):
        varin, vfac = vinfo['alias'], vinfo['fscl']
        season_full = ["DJF", "MAM", "JJA", "SON", "ANN"]
        season_list = (["MAM", "JJA", "SON", "ANN"]
                       if (skip_first_year_djf and year == self.years[0]) else season_full)

        for exp in self.model_list:
            ref = self.exp_dict[exp]['ref']
            out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_{metric}_{exp}_{year}.nc")
            if os.path.exists(out_file) and not self.force:
                print(f"Skipping {metric} for {var} in {exp} ({year}) — file exists.")
                continue
            if os.path.exists(out_file) and self.force:
                os.remove(out_file)

            bias_list, pval_list, sig_list = [], [], []
            mean_bias_list, rmse_list, pcorr_list = [], [], []

            for season in season_list:
                time_sub = (self._get_time_range_for_year(year)
                            if season == "ANN" else self._get_time_range_for_season(year, season))
                try:
                    obs = self._read_reference_data(
                        ref=ref, period=str(year), time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.ref_dict[ref]['run'], template=self.ref_template
                    )[varin].astype("float64")

                    ds = self._read_model_data(
                        exp=exp, period=str(year), time_sub=time_sub,
                        regnam=self.regnam, var=var, vfac=vfac,
                        data_dir=self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run']),
                        template=self.mod_template
                    )
                    obs = self._align_to_times(obs.to_dataset(name=varin), time_sub)[varin]
                    ds  = self._align_to_times(ds, time_sub)
                    fcst = ds[varin].astype("float64")
                except FileNotFoundError:
                    print(f"Data missing for {var}, {exp}, {year}, {season} — skipping.")
                    lat = xr.DataArray(np.full((1,), np.nan), dims=('lat',), coords={'lat':[np.nan]})
                    bias_list.append(xr.full_like(lat, np.nan).rename('bias').expand_dims(season=[season]))
                    pval_list.append(xr.full_like(lat, np.nan).rename('pvalue').expand_dims(season=[season]))
                    sig_list.append(xr.full_like(lat, 0).rename('sig_mask').expand_dims(season=[season]))
                    mean_bias_list += [np.nan]; rmse_list += [np.nan]; pcorr_list += [np.nan]
                    continue

                lat_name = next((d for d in ['lat','latitude','y'] if d in obs.dims), None)
                lon_name = next((d for d in ['lon','longitude','x'] if d in obs.dims), None)
                if lat_name is None or lon_name is None:
                    raise ValueError("Expected lat/lon dims not found in data.")

                obs  = obs.chunk({lat_name: -1, lon_name: -1, 'time': -1})
                fcst = fcst.chunk({lat_name: -1, lon_name: -1, 'time': -1})

                obs_zm_t  = obs.mean(lon_name)
                fcst_zm_t = fcst.mean(lon_name)
                obs_zm  = obs_zm_t.mean('time')
                fcst_zm = fcst_zm_t.mean('time')
                bias_zm = (fcst_zm - obs_zm).rename('bias')

                d_t   = (fcst_zm_t - obs_zm_t)
                N     = d_t.sizes.get('time', 1)
                mean_d = d_t.mean('time')
                std_d  = d_t.std('time', ddof=1)

                r1 = xr.apply_ufunc(
                    self._lag1_corr_np, d_t,
                    input_core_dims=[['time']], output_core_dims=[[]],
                    vectorize=True, dask='parallelized', output_dtypes=[float]
                ).clip(min=-0.99, max=0.99)
                Neff = (N * (1.0 - r1) / (1.0 + r1)).clip(min=2.0, max=float(N))
                sqrt_Neff = xr.apply_ufunc(np.sqrt, Neff, dask='parallelized', output_dtypes=[float])
                se   = xr.where(sqrt_Neff > 0, std_d / sqrt_Neff, np.nan)
                tstat = xr.where(se > 0, mean_d / se, np.nan)

                # elementwise df = max(1, Neff-1), dask-safe
                df = xr.apply_ufunc(
                    np.maximum, Neff - 1.0, 1.0,
                    dask='parallelized', output_dtypes=[float]
                )
                
                if t_dist is not None:
                    # two-sided p-value from Student-t, vectorized
                    pval = xr.apply_ufunc(
                        lambda t, df_: 2.0 * (1.0 - t_dist.cdf(np.abs(t), df=df_)),
                        tstat, df,
                        dask='parallelized', output_dtypes=[float]
                    )
                else:
                    # normal approximation fallback, vectorized
                    pval = xr.apply_ufunc(
                        lambda z: np.erfc(np.abs(z) / np.sqrt(2.0)),
                        tstat, dask='parallelized', output_dtypes=[float]
                    )
                    
                pval = pval.rename('pvalue')
                sig_mask = (pval < 0.05).astype('i1').rename('sig_mask')

                # summary metrics (cos-lat weights; collapse non-lat dims to avoid non-scalar results)
                w = self._coslat_weights_from_coords(obs[lat_name])

                # before correlation (just after computing obs_zm, fcst_zm, bias_zm)
                obs_1d  = self._collapse_except(obs_zm,  keep=[lat_name])
                fcst_1d = self._collapse_except(fcst_zm, keep=[lat_name])
                
                # pattern correlation across latitude
                if xs is not None:
                    try:
                        pcorr_da = xs.pearson_r(
                            obs_1d, fcst_1d,
                            dim=[lat_name],
                            weights=self._coslat_weights_from_coords(obs[lat_name]),
                            skipna=True
                        )
                        pcorr = self._to_scalar_float(pcorr_da)
                    except TypeError:
                        pcorr_da = xs.pearson_r(obs_1d, fcst_1d, dim=[lat_name], skipna=True)
                        pcorr = self._to_scalar_float(pcorr_da)
                else:
                    pcorr = float(self._weighted_pearson_1d(obs_1d, fcst_1d, self._coslat_weights_from_coords(obs[lat_name])))

                # (2) weighted means over latitude → still might have leftover dims if bias_zm has extra dims
                mb_da_lat   = bias_zm.weighted(w).mean(lat_name)
                rmse_da_lat = ((fcst_zm - obs_zm) ** 2).weighted(w).mean(lat_name) ** 0.5
                
                # (3) collapse any remaining dims to scalars
                mb_da   = self._to_scalar_da(mb_da_lat)
                rmse_da = self._to_scalar_da(rmse_da_lat)
                
                # (4) finalize Python floats
                mean_bias = self._to_scalar_float(mb_da)
                rmse      = self._to_scalar_float(rmse_da)
                
                bias_list.append(bias_zm.expand_dims(season=[season]))
                pval_list.append(pval.expand_dims(season=[season]))
                sig_list.append(sig_mask.expand_dims(season=[season]))
                mean_bias_list.append(mean_bias); rmse_list.append(rmse); pcorr_list.append(pcorr)

            ds_out = xr.Dataset(
                data_vars=dict(
                    bias_zonal   = xr.concat(bias_list, dim='season'),
                    pvalue       = xr.concat(pval_list, dim='season'),
                    sig_mask     = xr.concat(sig_list, dim='season'),
                    mean_bias    = xr.DataArray(np.asarray(mean_bias_list), dims="season", coords={"season": season_list}),
                    rmse         = xr.DataArray(np.asarray(rmse_list),      dims="season", coords={"season": season_list}),
                    pattern_corr = xr.DataArray(np.asarray(pcorr_list),     dims="season", coords={"season": season_list}),
                )
            )
            self._annotate_metadata(ds_out, var)
            ds_out.to_netcdf(out_file)
            print(f"Saved: {out_file}")

    # --------------------------- time helpers ---------------------------

    def _period_label(self):
        return f"{self.tstart[:7].replace('-','')}_{self.tend[:7].replace('-','')}"

    def _all_month_starts(self):
        y0, m0 = int(self.tstart[:4]), int(self.tstart[5:7])
        y1, m1 = int(self.tend[:4]),   int(self.tend[5:7])
        dates = []
        y, m = y0, m0
        while (y < y1) or (y == y1 and m <= m1):
            dates.append(DatetimeNoLeap(y, m, 1))
            m += 1
            if m == 13: m = 1; y += 1
        return xr.CFTimeIndex(dates)

    def _get_time_range_for_season_period(self, season):
        months, _ = self.seasons[season]
        all_months = self._all_month_starts()
        if season == "ANN":
            return all_months
        return xr.CFTimeIndex([d for d in all_months if d.month in months])

    def _get_time_range_for_year(self, year):
        freq_map = {"3hourly": "3h", "6hourly": "6h", "monthly": "1MS"}
        return xr.cftime_range(f"{year}-01-01", f"{year}-12-31",
                               freq=freq_map.get(self.frequency, "1MS"),
                               calendar="noleap")

    def _get_time_range_for_season(self, year, season):
        months, _ = self.seasons[season]
        dates = []
        for m in months:
            y = year if not (season == "DJF" and m == 12) else year - 1
            dates.append(DatetimeNoLeap(y, m, 1))
        return xr.CFTimeIndex(sorted(dates))

    def _years_from_time_sub(self, time_sub, fallback_year=None):
        years = set()
        if isinstance(time_sub, slice):
            for endpoint in (time_sub.start, time_sub.stop):
                if hasattr(endpoint, "year"): years.add(int(endpoint.year))
        else:
            try:
                for t in time_sub:
                    if hasattr(t, "year"): years.add(int(t.year))
            except TypeError:
                if hasattr(time_sub, "year"): years.add(int(time_sub.year))
        if not years and fallback_year is not None:
            years = {int(fallback_year)}
        return sorted(years)

    # ----------------------- IO + harmonization -----------------------

    def _read_model_data(self, exp, period, time_sub, regnam, var, vfac, data_dir, template, diag_print=False):
        years_needed = self._years_from_time_sub(time_sub, fallback_year=period)
        base, target_plev = self._parse_varname(var)
        if base == "Z": base = 'Z3'

        paths = []
        for y in years_needed:
            patt = self._render_pattern(template, y, var=base)
            pattern = os.path.join(data_dir, patt)
            matches = sorted(glob.glob(pattern))
            if not matches: print(f"[WARN] Model no matches: {pattern}")
            paths.extend(matches)
        if not paths:
            raise FileNotFoundError(f"No model files found for years={years_needed}")

        dm = xr.open_mfdataset(paths, combine="by_coords")

        (lat_bnds, lon_bnds) = self.define_region(regnam)
        lat_name = next((d for d in ['lat','latitude','y'] if d in dm.coords), None)
        lon_name = next((d for d in ['lon','longitude','x'] if d in dm.coords), None)
        if lat_name is None or lon_name is None:
            raise ValueError("Model data missing lat/lon coordinates.")

        dm = self._select_level(dm, target_plev)

        # wrap lon only if [0,360]
        dm = self._maybe_wrap_lon(dm, lon_name)

        # Decide whether to convert region bounds based on data coords
        lon = dm[lon_name]
        lon_min, lon_max = float(lon.min()), float(lon.max())
        
        # If the requested region is effectively global, skip lon slicing entirely
        global_like = (lon_bnds[0] <= -179.999) and (lon_bnds[1] >= 179.999)
        
        if not global_like:
            # Convert bounds only if data was 0–360 before wrapping
            # (After _maybe_wrap_lon the data is in [-180,180), so check original range by inference)
            # Heuristic: if min>=0 previously, wrapping would have created negatives.
            # A simpler robust rule: if current data is already in [-180,180), don't touch the bounds.
            # Here we keep bounds as-is unless you explicitly want to force-wrap.
            pass  # keep lon_bnds as provided

        dm = self._set_time_to_midpoint(dm)
        dm = self._normalize_time_to_month_start(dm)
        dm = dm.convert_calendar("noleap", use_cftime=True)

        # align strictly to requested monthly starts
        dm = self._align_to_times(dm, time_sub)
        
        # spatial slice (lat order agnostic)
        dm = self._smart_lat_slice(dm, lat_bnds, lat_name)
        if not global_like:
            dm = dm.sel({lon_name: slice(*lon_bnds)})
            
        if base in dm:
            dm[base] = self.apply_unit_scaling_mod(base, dm[base], vfac)

        if diag_print and "time" in dm.dims:
            print(f"[DIAG] Model Time range = {str(dm.time.min().values)} → {str(dm.time.max().values)} (n={dm.sizes['time']})")
        return dm

    def _read_reference_data(self, ref, period, time_sub, regnam, var, vfac, data_dir, template, diag_print=False):
        years_needed = self._years_from_time_sub(time_sub, fallback_year=period)
        base, target_plev = self._parse_varname(var)
        if base == "Z": base = 'Z3'

        paths = []
        for y in years_needed:
            patt = self._render_pattern(template, y, var=base)
            p = os.path.join(data_dir, patt)
            matches = sorted(glob.glob(p))
            if not matches: print(f"[WARN] Reference no matches: {p}")
            paths.extend(matches)
        if not paths:
            raise FileNotFoundError(f"No reference files found for years={years_needed}")

        dr = xr.open_mfdataset(paths, combine="by_coords")

        # rename coords if needed
        rename_dict = {}
        if "longitude" in dr.dims and "lon" not in dr.dims: rename_dict["longitude"] = "lon"
        if "latitude"  in dr.dims and "lat" not in dr.dims: rename_dict["latitude"]  = "lat"
        if rename_dict: dr = dr.rename(rename_dict)

        lat_name = next((d for d in ['lat','latitude','y'] if d in dr.coords), None)
        lon_name = next((d for d in ['lon','longitude','x'] if d in dr.coords), None)
        if lat_name is None or lon_name is None:
            raise ValueError("Reference data missing lat/lon coordinates.")

        dr = self._select_level(dr, target_plev)
        
        dr = self._maybe_wrap_lon(dr, lon_name)

        # Get region lon bounds here (define it!)
        (lat_bnds, lon_bnds) = self.define_region(regnam)
        
        # As above, skip lon slicing for global-like windows
        global_like = (lon_bnds[0] <= -179.999) and (lon_bnds[1] >= 179.999)


        lon = dm[lon_name] if "dm" in locals() else dr[lon_name]
        lon_min, lon_max = float(lon.min()), float(lon.max())

        dr = dr.convert_calendar("noleap", use_cftime=True)
        dr = self._normalize_time_to_month_start(dr)

        # apply time selection (keep order consistent with time_sub)
        if isinstance(time_sub, slice):
            dr = dr.sel(time=time_sub)
        else:
            try:
                dr = dr.sel(time=slice(*time_sub))
            except Exception:
                dr = dr.sel(time=time_sub)

        # spatial slice (lat order agnostic)
        (lat_bnds, _) = self.define_region(regnam)
        dr = self._smart_lat_slice(dr, lat_bnds, lat_name)
        if not global_like:
            dr = dr.sel({lon_name: slice(*lon_bnds)})
        # else: no lon slice (global)

        if base in dr:
            dr[base] = self.apply_unit_scaling_obs(base, dr[base], vfac)

        if diag_print and "time" in dr.dims:
            print(f"[DIAG] OBS Time range = {str(dr.time.min().values)} → {str(dr.time.max().values)} (n={dr.sizes['time']})")
        return dr

    # -------------------------- helpers & math --------------------------

    def _require_varinfo(self, var):
        if var not in self.var_dict:
            raise ValueError(f"Variable '{var}' is not defined in var_dict.")
        return self.var_dict[var]

    @staticmethod
    def _parse_varname(varname):
        m = re.match(r"([A-Za-z]+)(\d+)$", varname)
        if m:
            return m.group(1), int(m.group(2)) * 100  # hPa → Pa
        return varname, None

    @staticmethod
    def _select_level(dr, target_plev):
        if target_plev is None:
            return dr
        for lev_dim in ["plev", "lev", "level"]:
            if lev_dim in dr.dims:
                lev = dr[lev_dim]
                vals = lev.values
                units = getattr(lev, "units", "").lower()
                # normalize to hPa
                if "pa" in units and "hpa" not in units:
                    vals = vals / 100.0
                elif np.nanmax(vals) > 2000:
                    vals = vals / 100.0
                target_hpa = target_plev / 100.0
                # try linear interpolation in pressure
                if vals.size >= 2 and np.isfinite(vals).all():
                    dr = dr.assign_coords({lev_dim: vals})
                    try:
                        return dr.interp({lev_dim: target_hpa})
                    except Exception:
                        pass
                idx = int(np.nanargmin(np.abs(vals - target_hpa)))
                return dr.isel({lev_dim: idx}, drop=True)
        return dr

    @staticmethod
    def _coslat_weights_from_coords(lat):
        w = np.cos(np.deg2rad(lat))
        return xr.where(np.isfinite(w), w, 0.0)

    @staticmethod
    def _weighted_pearson_1d(a, b, w):
        a, b, w = xr.align(a, b, w, join="inner")
        mask = xr.apply_ufunc(np.isfinite, a) & xr.apply_ufunc(np.isfinite, b) & (w > 0)
        a = a.where(mask); b = b.where(mask); w = w.where(mask)
        ws = w.sum()
        if float(ws) <= 0: return np.nan
        mu_a = (a*w).sum()/ws; mu_b = (b*w).sum()/ws
        da, db = a-mu_a, b-mu_b
        cov = (w*da*db).sum()/ws
        va  = (w*da*da).sum()/ws
        vb  = (w*db*db).sum()/ws
        denom = np.sqrt(float(va)*float(vb))
        return float(cov)/denom if denom > 0 else np.nan

    @staticmethod
    def _set_time_to_midpoint(ds):
        if 'time_bnds' in ds.variables and 'time' in ds.coords:
            mid = ds['time_bnds'].mean(dim=ds['time_bnds'].dims[-1])
            ds = ds.assign_coords(time=mid)
        return ds

    @staticmethod
    def _normalize_time_to_month_start(ds):
        if "time" not in ds.coords: return ds
        new_times = []
        for t in ds["time"].values:
            t_str = str(type(t))
            if "datetime64" in t_str:
                import pandas as pd
                ts = pd.to_datetime(t)
                new_times.append(ts.to_period('M').to_timestamp('MS').to_pydatetime())
            elif "Timestamp" in t_str:
                new_times.append(t.to_period('M').to_timestamp('MS').to_pydatetime())
            else:
                cls = t.__class__
                new_times.append(cls(t.year, t.month, 1))
        return ds.assign_coords(time=("time", new_times))

    # --- NEW: strict time alignment & spatial helpers ---

    def _align_to_times(self, ds, target_index, *, tolerance_days=3):
        if "time" not in ds.coords:
            return ds
        ds2 = ds.reindex(
            time=target_index,
            method="nearest",
            tolerance=np.timedelta64(tolerance_days, "D")
        )
        # ensure no missing reindexes
        if np.isnan(ds2["time"].astype("datetime64[ns]").values).any():
            raise ValueError("[ERROR] Failed strict month-start alignment (missing after reindex).")
        return ds2

    @staticmethod
    def _to_m180_180(lon):
        return ((lon + 180) % 360) - 180

    def _maybe_wrap_lon(self, ds, lon_name):
        lon = ds[lon_name]
        if lon.min() >= 0 and lon.max() <= 360:
            ds = ds.assign_coords({lon_name: self._to_m180_180(lon)}).sortby(lon_name)
        return ds

    def _smart_lat_slice(self, arr, lat_bnds, lat_name):
        lo, hi = lat_bnds
        lat = arr[lat_name]
        asc = bool(lat.values[0] < lat.values[-1])
        return arr.sel({lat_name: slice(lo, hi) if asc else slice(hi, lo)})

    @staticmethod
    def apply_unit_scaling_obs(var, da, vfac):
        # NOTE: ensure both model & obs end up in the same units.
        if var in ['Z', 'Z3']:
            return da * vfac / 9.80616
        elif var in ['SHFLX', 'TAUX', 'TAUY']:
            return da * vfac * -1.0
        elif var == 'LHFLX':
            # If obs are in kg m^-2 s^-1 (evap), multiply by Lv -> W m^-2 and sign convention if needed
            return da * vfac * -2.501e6
        elif var in ['T', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        elif var == 'PRECT':
            # convert to mm/day
            return da * vfac / 3600.0 * 1000.0 * 86400.0
        else:
            return da * vfac

    @staticmethod
    def apply_unit_scaling_mod(var, da, vfac):
        if var in ['Z', 'Z3']:
            return da * vfac / 9.80616
        elif var == 'PRECT':
            return da * vfac * 1000.0 * 86400.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        else:
            return da * vfac

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global':   [(-90, 90),  (-180, 180)],
            'Atlantic': [(5, 55),    (-95, -40)],
            'CONUS':    [(25, 50),   (-125, -95)],
            'Antarctic':[(-90, -50), (-180, 180)],
            'PolarN':   [(50, 90),   (-180, 180)],
            'Greenland':[(60, 85),   (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Unknown region '{regnam}'. Valid: {list(reg_dict)}")
        return reg_dict[regnam]

    def extract_var_list(self):
        return {
            'U': {'alias': 'U',  'unit': 'm s$^{-1}$', 'fscl': 1.0, 'min': -0.2,  'max': 0.2,   'nlev': 11},
            'V': {'alias': 'V',  'unit': 'm s$^{-1}$', 'fscl': 1.0, 'min': -0.2,  'max': 0.2,   'nlev': 11},
            'T': {'alias': 'T',  'unit': '°C',         'fscl': 1.0, 'min': -10,   'max': 10,    'nlev': 11},
            'Q': {'alias': 'Q',  'unit': 'kg kg$^{-1}$','fscl':1.0, 'min': -2e-3, 'max': 2e-3,  'nlev': 11},
            'Z': {'alias': 'Z3', 'unit': 'm',          'fscl': 1.0, 'min': -200,  'max': 200,   'nlev': 11},
        }

    # --- rendering helper for filename patterns ---
    @staticmethod
    def _render_pattern(template, year, var=None):
        # allow {year}, {var}; fall back to single {} for {year}
        try:
            return template.format(year=year, var=var)
        except Exception:
            try:
                return template.format(year)
            except Exception:
                # legacy %Y/%VAR% support
                patt = template.replace("%Y", str(year))
                if var is not None:
                    patt = patt.replace("%VAR%", str(var))
                return patt
                
    @staticmethod
    def _lag1_corr_np(a):
        """Lag-1 autocorrelation of a 1D array; NaN-safe; returns float."""
        a = np.asarray(a)
        good = np.isfinite(a)
        a = a[good]
        if a.size < 3:
            return np.nan
        a0 = a[:-1] - a[:-1].mean()
        a1 = a[1:]  - a[1:].mean()
        denom = np.sqrt((a0**2).sum() * (a1**2).sum())
        return (a0*a1).sum() / denom if denom > 0 else np.nan

    @staticmethod
    def _annotate_metadata(ds_out, var):
        ds_out['bias_zonal'].attrs['long_name'] = f"Zonal-mean bias of {var} (fcst - obs)"
        ds_out['pvalue'].attrs['description']   = "Two-sided p-value for H0: mean monthly bias == 0 (pooled within season), AR(1) Neff"
        ds_out['sig_mask'].attrs['note']        = "1 = significant at 0.05 level"
        ds_out['mean_bias'].attrs['description']= "Area-weighted mean bias over latitude (cos-lat)"
        ds_out['rmse'].attrs['description']     = "Area-weighted RMSE over latitude (cos-lat)"
        ds_out['pattern_corr'].attrs['description'] = "Pattern correlation across latitude (zonal means)"
        if 'n_months' in ds_out:
            ds_out['n_months'].attrs['description'] = "Number of monthly samples used in the season"
            

In [7]:
if __name__ == "__main__":
    # --- paths ---
    top_path  = "/pscratch/sd/z/zhan391/seacrogs_scratch"
    data_path = f"{top_path}/post_data"
    out_path  = "/pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias"
    os.makedirs(out_path, exist_ok=True)

    # --- experiments metadata ---
    exp_json = f"{data_path}/scripts/ml_exp_info.json"
    with open(exp_json, "r") as f:
        exp_dict = json.load(f)

    # ensure each experiment has a 'ref' key (default to ERA5 if missing)
    for k, v in exp_dict.items():
        if "ref" not in v:
            v["ref"] = "ERA5"

    # --- time / freq / region ---
    # Use full dates; class only uses the year part for looping,
    # but full dates are safer for future extensions.
    tstart = "2012-01-01"
    tend   = "2014-12-31"
    freq   = "monthly"
    regnam = "global"

    # --- reference dataset info ---
    ref_dict = {
        "ERA5": {
            "run": f"{data_path}/ERA5",
            "period": "200801_201712",
        }
    }

    # --- model file root ---
    # The class will look under: path_in.replace("%(CASENAME)", exp_dict[exp]["run"])
    # and use the internal template "{}_*.nc". Adjust templates in the class if needed.
    path_template = f"{data_path}/%(CASENAME)/{freq}"

    # --- variables ---
    # None → use defaults from extract_var_list(); or set e.g. ['U200','T','Z']
    variables = None

    # --- init calculator ---
    calculator = ErrorMetricCalculator(
        regnam=regnam,
        tstart=tstart,
        tend=tend,
        frequency=freq,
        model_list=list(exp_dict.keys()),
        ref_dict=ref_dict,
        exp_dict=exp_dict,
        path_in=path_template,
        out_path=out_path,
        var_list=variables,
        force=True,
    )

    # --- run (single label used in output filenames) ---
    calculator.compute(metric="zonal_bias")
    

Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/U_global_zonal_bias_CLIM_201201_201412.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/U_global_zonal_bias_UNet_201201_201412.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/U_global_zonal_bias_UNetMP_201201_201412.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/U_global_zonal_bias_IUNet_201201_201412.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/U_global_zonal_bias_MnM_201201_201412.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/V_global_zonal_bias_CLIM_201201_201412.nc
Saved: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/zonal_bias/monthly/V_global_zonal_bias_UNet_